# Phase 1 (Merged) — Zillow + FRED Ingest, Join, and Coverage Audit

**Merges two independently-built Phase 1 pipelines** (Michel's SQL/Postgres variant and Jolenn's pandas-based repo) into the single version this team repo carries forward. Four decisions were made comparing the two, each recorded here so the reasoning isn't lost:

1. **Zillow NaN handling — adopts Jolenn's approach.** The SQL-variant repo used a blanket `dropna(subset=["zhvi"])`, which silently removes genuine *internal* reporting gaps along with the expected leading/trailing edge months. Jolenn's repo trims each county to its own `[first_valid, last_valid]` window but *keeps* internal gaps as NULL `zhvi` rows -- more information-preserving, and the reason her `zhvi_monthly` has 709,996 rows against the SQL-variant's 708,953 (a difference of exactly 1,043 rows).
2. **Month storage — adopts month-end.** The SQL-variant repo normalized `month` to month-*start* via `.dt.to_period("M").dt.to_timestamp()`. Jolenn's repo parses Zillow's own column headers directly, keeping them at month-*end* -- Zillow's native format, and already what `mortgage_monthly` uses after resampling. Adopting month-end here removes the need for the `date_trunc('month', ...)` fix the SQL-variant's join previously required -- a fix made necessary by that repo's own normalization choice, not by any inherent Zillow/FRED mismatch.
3. **Correction to a prior assumption.** `sql/002_join_check.sql` previously attributed the 708,953-vs-709,996 gap to "stale rows from an earlier run silently accumulating." That explanation doesn't hold up: Jolenn's pipeline clears the table (`DELETE FROM`) before every load and the table has a `(county_fips, month)` primary key, so duplicate accumulation isn't possible there. The real cause is decision 1 above, not a bug in either pipeline. The corrected `002_join_check.sql` below reflects this.
4. **The join itself stays in SQL** -- the SQL-variant repo's own differentiator versus the pandas repo -- now as a **plain equality join** rather than a `date_trunc`-based one, since both tables share the same month-end convention after decision 2.

FRED's resampling convention (`.mean()` of each month's weekly readings) was already identical between both repos and is unchanged here.

**Expects** `sql/001_schema.sql` already applied to this team's database (creates `zhvi_monthly` and `mortgage_monthly`), and the corrected `sql/002_join_check.sql` (below, and provided alongside this notebook) present in `sql/`. Connection details come from `.env` in the repo root.

## 1. Setup & Connection

In [2]:
import os
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

pd.set_option("display.max_columns", 20)
load_dotenv(override=True)

DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD", "")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = os.environ.get("PGPORT", "5432")
# New team database -- deliberately distinct from either personal repo's
# own database name (capstone_sql_variant / housing_decline), so a stray
# .env pointing at a personal repo's DB is caught immediately rather than
# silently writing into the wrong database.
DB_NAME = os.environ.get("PGDATABASE", "housing_risk_team")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), version();")).fetchone())

('housing_risk_team', 'PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit')


**Expected output:** `('housing_risk_team', 'PostgreSQL ...')`. If this shows a different database name, stop and fix `.env` before continuing -- every step after this loads real data into whatever database this cell connects to.

## 2. Zillow ZHVI Ingest

In [3]:
ZILLOW_URL = (
    "https://files.zillowstatic.com/research/public_csvs/zhvi/"
    "County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)

RAW_DIR = "../data/raw/zillow"
os.makedirs(RAW_DIR, exist_ok=True)
raw_path = os.path.join(RAW_DIR, "zhvi_county.csv")

if not os.path.exists(raw_path):
    resp = requests.get(ZILLOW_URL, timeout=60)
    resp.raise_for_status()
    with open(raw_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {raw_path}")
else:
    print(f"Using cached file at {raw_path}")

zhvi_wide = pd.read_csv(raw_path)
assert set(zhvi_wide["RegionType"].unique()) == {"county"}, "Not a county-only file!"
print("Raw shape:", zhvi_wide.shape)

Downloaded 13,579,570 bytes to ../data/raw/zillow\zhvi_county.csv
Raw shape: (3071, 329)


## 3. Reshape: wide → long, build county FIPS, trim to each county's coverage window

Three transforms, same as Jolenn's `01_zillow_ingest_clean.ipynb`:

1. **Melt** every date-like column into `month` / `zhvi` rows. `month` is parsed directly from Zillow's own column headers with `pd.to_datetime()` -- **not** routed through `.dt.to_period("M").dt.to_timestamp()`, which would silently normalize every date to month-*start* and reintroduce the mismatch against `mortgage_monthly`'s month-end convention.
2. **Concatenate `StateCodeFIPS` + `MunicipalCodeFIPS`** into `county_fips` -- identical in both source repos.
3. **Trim each county to its own `[first_valid, last_valid]` window**, instead of a blanket `dropna(subset=["zhvi"])`. A county with no ZHVI before Zillow started covering it (or after it stopped) is outside that county's reporting window and is correctly dropped. A NaN that falls *inside* the window, with real values both before and after, is a genuine gap and is kept as a NULL `zhvi` row -- the same way a month missing from `mortgage_monthly` surfaces as NULL rather than a dropped row in the join.

In [4]:
id_cols = [c for c in zhvi_wide.columns if not c[:2].isdigit()]
date_cols = [c for c in zhvi_wide.columns if c not in id_cols]

zhvi_long = zhvi_wide.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="month",
    value_name="zhvi",
)

# Zillow's column headers are already exact month-end dates (e.g.
# "2000-01-31", "2000-02-29"). Parsed as-is -- see the markdown above for
# why this must NOT be routed through .dt.to_period("M").dt.to_timestamp().
zhvi_long["month"] = pd.to_datetime(zhvi_long["month"])

# Zero-pad: state is 2 digits, county is 3.
zhvi_long["county_fips"] = (
    zhvi_long["StateCodeFIPS"].astype(str).str.zfill(2)
    + zhvi_long["MunicipalCodeFIPS"].astype(str).str.zfill(3)
)

zhvi_clean = zhvi_long.rename(columns={"RegionName": "region_name", "State": "state"})[
    ["county_fips", "month", "zhvi", "region_name", "state"]
]

# Trim each county to its own [first published, last published] window,
# keeping any internal NaN gap as a NULL zhvi row rather than dropping it.
valid = zhvi_clean.dropna(subset=["zhvi"])
coverage_bounds = (
    valid.groupby("county_fips")["month"]
    .agg(first_valid="min", last_valid="max")
)
zhvi_clean = zhvi_clean.merge(coverage_bounds, on="county_fips", how="left")
zhvi_clean = zhvi_clean[
    (zhvi_clean["month"] >= zhvi_clean["first_valid"])
    & (zhvi_clean["month"] <= zhvi_clean["last_valid"])
].drop(columns=["first_valid", "last_valid"])

n_internal_gaps = zhvi_clean["zhvi"].isna().sum()
print("Cleaned shape:", zhvi_clean.shape)
print("Unique counties:", zhvi_clean["county_fips"].nunique())
print("Date range:", zhvi_clean["month"].min().date(), "to", zhvi_clean["month"].max().date())
print(f"Internal NULL zhvi rows kept (genuine gaps, not edge trim): {n_internal_gaps:,}")
zhvi_clean.head(3)

Cleaned shape: (709996, 5)
Unique counties: 3071
Date range: 2000-01-31 to 2026-08-31
Internal NULL zhvi rows kept (genuine gaps, not edge trim): 1,043


,county_fips,month,zhvi,region_name,state
0,06037,2000-01-31,210217.606330,Los Angeles County,CA
1,17031,2000-01-31,152726.258141,Cook County,IL
2,48201,2000-01-31,113021.492070,Harris County,TX


**Expect roughly 709,996 rows** (may drift slightly month to month as Zillow's file keeps extending forward) -- noticeably more than the SQL-variant repo's old 708,953, by design: those ~1,043 extra rows are the internal gaps this version correctly keeps instead of dropping.

## 4. Coverage Audit

Per-county check: flags counties with under 6 months of history (can't support the 6-month momentum lag feature yet) and counties with zero presence in the training period. Diagnostic only -- doesn't drop rows. Identical logic in both source repos; unaffected by the NaN-handling decision above, since it operates on first/last month per county, not on internal gap rows.

In [5]:
county_month_counts = (
    zhvi_clean.groupby("county_fips")["month"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "first_month", "max": "last_month", "count": "n_months"})
)

n_counties = county_month_counts.shape[0]
n_short_history = (county_month_counts["n_months"] < 6).sum()

train_cutoff = pd.Timestamp("2021-09-30")
training_presence = (
    zhvi_clean[zhvi_clean["month"] <= train_cutoff]
    .groupby("county_fips")["month"]
    .count()
)
n_no_training_presence = n_counties - training_presence.shape[0]

print(f"Counties: {n_counties:,}")
print(f"Total county-month rows: {len(zhvi_clean):,}")
print(f"Counties with < 6 months of history: {n_short_history:,}")
print(f"Counties with zero rows in the training period: {n_no_training_presence:,}")

Counties: 3,071
Total county-month rows: 709,996
Counties with < 6 months of history: 0
Counties with zero rows in the training period: 50


**Expect:** 3,071 counties, 0 short-history counties, 50 counties with zero training-period presence -- both source repos agreed on these two figures exactly, despite their differing total row counts.

## 5. Load Zillow to Postgres

`TRUNCATE` before load (not just `if_exists="append"` alone), so this cell is safe to re-run top to bottom any number of times without a `UniqueViolation` on `(county_fips, month)` or stale rows lingering from an earlier run.

In [6]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE zhvi_monthly;"))

zhvi_clean.to_sql(
    "zhvi_monthly",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)

with engine.connect() as conn:
    loaded = conn.execute(text("SELECT COUNT(*) FROM zhvi_monthly;")).scalar()
print(f"zhvi_monthly now has {loaded:,} rows")
assert loaded == len(zhvi_clean), "Row count in Postgres doesn't match the cleaned frame."

zhvi_monthly now has 709,996 rows


## 6. FRED `MORTGAGE30US` Ingest

Unchanged from both source repos -- `.mean()` of each month's weekly readings, resampled to month-end (`"ME"`). Already identical between the SQL-variant and pandas repos; no decision needed here.

In [7]:
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

RAW_DIR = "../data/raw/fred"
os.makedirs(RAW_DIR, exist_ok=True)
raw_path = os.path.join(RAW_DIR, "mortgage30us.csv")

if not os.path.exists(raw_path):
    resp = requests.get(FRED_URL, timeout=60)
    resp.raise_for_status()
    with open(raw_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content):,} bytes to {raw_path}")
else:
    print(f"Using cached file at {raw_path}")

rate_weekly = pd.read_csv(raw_path)
rate_weekly.columns = ["date", "mortgage_rate"]
rate_weekly["date"] = pd.to_datetime(rate_weekly["date"])
rate_weekly["mortgage_rate"] = pd.to_numeric(rate_weekly["mortgage_rate"], errors="coerce")
rate_weekly = rate_weekly.dropna(subset=["mortgage_rate"])
print("Raw weekly shape:", rate_weekly.shape)

rate_monthly = (
    rate_weekly.set_index("date")["mortgage_rate"]
    .resample("ME")
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"date": "month", "mean": "mortgage_rate", "count": "n_weeks"})
)
rate_monthly = rate_monthly.dropna(subset=["mortgage_rate"])
print("Monthly shape:", rate_monthly.shape)
rate_monthly.tail(3)

Downloaded 46,931 bytes to ../data/raw/fred\mortgage30us.csv
Raw weekly shape: (2896, 2)
Monthly shape: (666, 3)


,month,mortgage_rate,n_weeks
663,2026-07-31,6.5420,5
664,2026-08-31,6.6675,4
665,2026-09-30,6.8625,4


## 7. Load FRED to Postgres

In [8]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE mortgage_monthly;"))

rate_monthly[["month", "mortgage_rate"]].to_sql(
    "mortgage_monthly",
    engine,
    if_exists="append",
    index=False,
)

with engine.connect() as conn:
    loaded = conn.execute(text("SELECT COUNT(*) FROM mortgage_monthly;")).scalar()
print(f"mortgage_monthly now has {loaded:,} rows")
assert loaded == len(rate_monthly), "Row count in Postgres doesn't match the cleaned frame."

mortgage_monthly now has 666 rows


## 8. Join — in SQL, now a plain equality join

Runs `sql/002_join_check.sql` (corrected version, provided alongside this notebook) on the Postgres server. Because both tables now share the same month-end convention (Section 3's decision), the join no longer needs `date_trunc('month', ...)` on either side -- a plain `m.month = z.month` is sufficient and slightly cheaper to plan/execute.

In [9]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS county_month_joined;"))
    conn.execute(text(open("../sql/002_join_check.sql").read()))

joined = pd.read_sql("SELECT * FROM county_month_joined;", engine)
print("Joined shape:", joined.shape)
print("Rows with missing mortgage_rate:", joined["mortgage_rate"].isna().sum())

counties_before = set(zhvi_clean["county_fips"].unique())
counties_after = set(joined["county_fips"].unique())
dropped = len(counties_before - counties_after)
print("Counties dropped during join:", dropped)
assert dropped == 0, "A county was dropped during the join."
assert len(joined) == len(zhvi_clean), "Row count changed across the join."
joined.head(3)

Joined shape: (709996, 6)
Rows with missing mortgage_rate: 0
Counties dropped during join: 0


,county_fips,month,zhvi,region_name,state,mortgage_rate
0,06037,2000-01-31,210217.606330,Los Angeles County,CA,8.21
1,17031,2000-01-31,152726.258141,Cook County,IL,8.21
2,48201,2000-01-31,113021.492070,Harris County,TX,8.21


## 9. Provenance Check

Reproduces the Train/Embargo/Test positive-rate figures from `docs/phase0_framing.md`'s Data Provenance table, against this merged pipeline's own `county_month_joined` output -- confirms the join and the NaN-handling change above didn't move these figures in any meaningful way (they're computed at quarter-end grain, so the internal monthly NULL rows kept in Section 3 should have little to no effect unless one happens to land on a quarter-end month for some county).

In [10]:
joined["month"] = pd.to_datetime(joined["month"])
joined["quarter"] = joined["month"].dt.to_period("Q")

zillow_q = (
    joined.sort_values("month")
    .groupby(["county_fips", "quarter"], as_index=False)
    .last()[["county_fips", "quarter", "zhvi"]]
    .sort_values(["county_fips", "quarter"])
    .reset_index(drop=True)
)
zillow_q["pct_change"] = zillow_q.groupby("county_fips")["zhvi"].pct_change()
zillow_q["decline_label"] = (zillow_q["pct_change"] < -0.02).astype(int)
zillow_q_valid = zillow_q.dropna(subset=["pct_change"]).copy()

def window_stats(mask, label):
    sub = zillow_q_valid[mask]
    rate = sub["decline_label"].mean() * 100
    print(f"{label:45s} rows={len(sub):>7,}   positive_rate={rate:.2f}%")

window_stats(pd.Series(True, index=zillow_q_valid.index), "Full history")
window_stats(zillow_q_valid["quarter"] <= pd.Period("2021Q3"), "Train (through 2021-09-30)")
window_stats(zillow_q_valid["quarter"] == pd.Period("2021Q4"), "Embargoed (2021-10-01 to 2021-12-31)")
window_stats(
    (zillow_q_valid["quarter"] >= pd.Period("2022Q1")) & (zillow_q_valid["quarter"] <= pd.Period("2026Q1")),
    "Test (2022-01-01 to 2026-03-31)",
)

Full history                                  rows=234,983   positive_rate=5.57%
Train (through 2021-09-30)                    rows=173,734   positive_rate=4.93%
Embargoed (2021-10-01 to 2021-12-31)          rows=  3,020   positive_rate=0.96%
Test (2022-01-01 to 2026-03-31)               rows= 52,087   positive_rate=8.15%


**Expected (per `docs/phase0_framing.md`):** Train ≈ 4.96%, Embargo ≈ 0.99%, Test ≈ 8.15%, Full history ≈ 5.59%. "Full history" will differ slightly run-to-run since Zillow's file keeps extending forward each month -- expected drift, not a bug. If Train/Embargo/Test move away from these figures by more than a rounding amount, stop and check whether an internal-gap row happened to fall on a quarter-end month for some county before proceeding to Phase 2.

## Status

Phase 1 merge complete. Adopts Jolenn's NaN-preserving Zillow cleaning and month-end date convention; keeps the SQL-variant repo's SQL-based join, simplified to a plain equality join now that both tables share one date convention. `sql/002_join_check.sql`'s misleading "stale-duplicate" comment is corrected in the version provided alongside this notebook. Run this notebook top to bottom against a fresh `housing_risk_team` database (schema applied from `sql/001_schema.sql` first), then run `sql/phase1_validation.sql` (provided separately) to confirm the load before moving to Phase 2.